### Data Ingestion


In [6]:
import time 
from tqdm import tqdm 
for i in tqdm(range(100)):
    time.sleep(1)

100%|██████████| 100/100 [01:40<00:00,  1.01s/it]


In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/Users/patel_parthk/Desktop/Project/Yoga-suggestion/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/patel_parthk/Desktop/Project/Yoga-suggestion/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf")
all_pdf_documents

Found 6 PDF files to process

Processing: Common Yoga Protocol Book-English.pdf
  ✓ Loaded 62 pages

Processing: Asana.pdf
  ✓ Loaded 557 pages

Processing: Yoga_Beginner.pdf
  ✓ Loaded 288 pages

Processing: LeslieKaminoffYogaAnatomyzliborg-200817-225010.pdf
  ✓ Loaded 233 pages

Processing: yoga.pdf
  ✓ Loaded 44 pages

Processing: YogaXII.pdf
  ✓ Loaded 131 pages

Total documents loaded: 1315


[Document(metadata={'producer': '', 'creator': 'PyPDF', 'creationdate': '2024-06-13T16:28:51+05:30', 'source': '../data/pdf/Common Yoga Protocol Book-English.pdf', 'total_pages': 62, 'page': 0, 'page_label': '1', 'source_file': 'Common Yoga Protocol Book-English.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': '', 'creator': 'PyPDF', 'creationdate': '2024-06-13T16:28:51+05:30', 'source': '../data/pdf/Common Yoga Protocol Book-English.pdf', 'total_pages': 62, 'page': 1, 'page_label': '2', 'source_file': 'Common Yoga Protocol Book-English.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': '', 'creator': 'PyPDF', 'creationdate': '2024-06-13T16:28:51+05:30', 'source': '../data/pdf/Common Yoga Protocol Book-English.pdf', 'total_pages': 62, 'page': 2, 'page_label': '3', 'source_file': 'Common Yoga Protocol Book-English.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': '', 'creator': 'PyPDF', 'creationdate': '202

In [17]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [18]:
chunks=split_documents(all_pdf_documents)
chunks

Split 1315 documents into 2450 chunks

Example chunk:
Content: Asana Pranayama 
Mudra Bandha 
Swami Satyananda Saraswati 
'og.t Pubhcatlons J ntst, Mungct, B1h.u, JndJ.J...
Metadata: {'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'PFU ScanSnap Manager 4.2.14', 'creationdate': '2012-05-24T20:23:43+08:00', 'moddate': '2021-05-07T15:35:13+02:00', 'source': '../data/pdf/Asana.pdf', 'total_pages': 557, 'page': 0, 'page_label': '1', 'source_file': 'Asana.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'PFU ScanSnap Manager 4.2.14', 'creationdate': '2012-05-24T20:23:43+08:00', 'moddate': '2021-05-07T15:35:13+02:00', 'source': '../data/pdf/Asana.pdf', 'total_pages': 557, 'page': 0, 'page_label': '1', 'source_file': 'Asana.pdf', 'file_type': 'pdf'}, page_content="Asana Pranayama \nMudra Bandha \nSwami Satyananda Saraswati \n'og.t Pubhcatlons J ntst, Mungct, B1h.u, JndJ.J"),
 Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'PFU ScanSnap Manager 4.2.14', 'creationdate': '2012-05-24T20:23:43+08:00', 'moddate': '2021-05-07T15:35:13+02:00', 'source': '../data/pdf/Asana.pdf', 'total_pages': 557, 'page': 1, 'page_label': '2', 'source_file': 'Asana.pdf', 'file_type': 'pdf'}, page_content='Asana Pranayama \nMudra Bandha \nWith kind regards, \\!a and prem \n�!{,�A*-'),
 Document(metadata={'producer': '3-Heights(

## Embedding

In [19]:
import numpy as np
import os
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb. config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics. pairwise import cosine_similarity

In [20]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2141.90it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


## VectorStore



In [21]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 7350


In [22]:
chunks

[Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'PFU ScanSnap Manager 4.2.14', 'creationdate': '2012-05-24T20:23:43+08:00', 'moddate': '2021-05-07T15:35:13+02:00', 'source': '../data/pdf/Asana.pdf', 'total_pages': 557, 'page': 0, 'page_label': '1', 'source_file': 'Asana.pdf', 'file_type': 'pdf'}, page_content="Asana Pranayama \nMudra Bandha \nSwami Satyananda Saraswati \n'og.t Pubhcatlons J ntst, Mungct, B1h.u, JndJ.J"),
 Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'PFU ScanSnap Manager 4.2.14', 'creationdate': '2012-05-24T20:23:43+08:00', 'moddate': '2021-05-07T15:35:13+02:00', 'source': '../data/pdf/Asana.pdf', 'total_pages': 557, 'page': 1, 'page_label': '2', 'source_file': 'Asana.pdf', 'file_type': 'pdf'}, page_content='Asana Pranayama \nMudra Bandha \nWith kind regards, \\!a and prem \n�!{,�A*-'),
 Document(metadata={'producer': '3-Heights(

In [23]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 2450 texts...


Batches: 100%|██████████| 77/77 [00:13<00:00,  5.69it/s]


Generated embeddings with shape: (2450, 384)
Adding 2450 documents to vector store...
Successfully added 2450 documents to vector store
Total documents in collection: 9800


In [25]:
### Pinecone Vector Store - One-time ingestion
import os
import uuid
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

class PineconeVectorStore:
    """Pinecone vector store that follows ChromaDB interface"""
    
    def __init__(self, index_name="yoga-rag", dimension=384):
        self.index_name = index_name
        self.dimension = dimension
        self.api_key = os.getenv("PINECONE_API_KEY")
        
        if not self.api_key:
            raise ValueError("PINECONE_API_KEY not found in environment variables")
        
        # Initialize Pinecone
        self.client = Pinecone(api_key=self.api_key)
        
        # Create index if doesn't exist
        if index_name not in self.client.list_indexes().names():
            self.client.create_index(
                name=index_name,
                dimension=dimension,
                metric="cosine",
                spec=ServerlessSpec(cloud="aws", region="us-east-1")
            )
            print(f"Created new Pinecone index: {index_name}")
        else:
            print(f"Using existing index: {index_name}")
        
        self.index = self.client.Index(index_name)
        stats = self.index.describe_index_stats()
        print(f"Vectors in index: {stats.total_vector_count}")
    
    def add_documents(self, documents, embeddings):
        """Add documents to Pinecone"""
        print(f"Adding {len(documents)} documents to Pinecone...")
        
        vectors = []
        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            
            # Clean metadata - Pinecone only accepts strings, numbers, booleans
            metadata = {
                "source_file": str(doc.metadata.get("source_file", "")),
                "file_type": str(doc.metadata.get("file_type", "")),
                "page": doc.metadata.get("page", 0),
                "text": doc.page_content[:1000]  # Store first 1000 chars
            }
            
            vectors.append({
                "id": doc_id,
                "values": emb.tolist(),
                "metadata": metadata
            })
        
        # Upsert in batches
        batch_size = 100
        for i in range(0, len(vectors), batch_size):
            batch = vectors[i:i + batch_size]
            self.index.upsert(vectors=batch)
            print(f"  Upserted batch {i//batch_size + 1}")
        
        print(f"Successfully added {len(documents)} documents!")
        
    def query(self, query_embeddings, n_results=5):
        """Query for similar documents"""
        results = self.index.query(
            vector=query_embeddings[0],
            top_k=n_results,
            include_metadata=True
        )
        
        documents = []
        metadatas = []
        distances = []
        ids = []
        
        for match in results.matches:
            ids.append(match.id)
            documents.append(match.metadata.get("text", "") if match.metadata else "")
            metadatas.append(match.metadata or {})
            distances.append(1 - match.score)  # Convert score to distance
            
        return {
            "documents": [documents],
            "metadatas": [metadatas],
            "distances": [distances],
            "ids": [ids]
        }
    
    @property
    def collection(self):
        return self
    
    def count(self):
        return self.index.describe_index_stats().total_vector_count


### Initialize Pinecone and ingest data
# Make sure PINECONE_API_KEY is set in your .env file
pinecone_store = PineconeVectorStore(index_name="yoga-rag", dimension=384)

### Ingest chunks into Pinecone (run this once)
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)
pinecone_store.add_documents(chunks, embeddings)

print("\n✅ Data ingestion complete!")
print(f"Total vectors in Pinecone: {pinecone_store.count()}")

Created new Pinecone index: yoga-rag
Vectors in index: 0
Generating embeddings for 2450 texts...


Batches: 100%|██████████| 77/77 [00:15<00:00,  5.10it/s]


Generated embeddings with shape: (2450, 384)
Adding 2450 documents to Pinecone...
  Upserted batch 1
  Upserted batch 2
  Upserted batch 3
  Upserted batch 4
  Upserted batch 5
  Upserted batch 6
  Upserted batch 7
  Upserted batch 8
  Upserted batch 9
  Upserted batch 10
  Upserted batch 11
  Upserted batch 12
  Upserted batch 13
  Upserted batch 14
  Upserted batch 15
  Upserted batch 16
  Upserted batch 17
  Upserted batch 18
  Upserted batch 19
  Upserted batch 20
  Upserted batch 21
  Upserted batch 22
  Upserted batch 23
  Upserted batch 24
  Upserted batch 25
Successfully added 2450 documents!

✅ Data ingestion complete!
Total vectors in Pinecone: 2450


### Retriever Pipeline From VectorStore

In [26]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int =5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [ ]:
rag_retriever.retrieve("what is my yoga")


Retrieving documents for query: 'what is my yoga'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.36it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_fb96f146_4',
  'content': 'Reprinted 2002, 2004 (twice), 2005, 2006 \nFourth (revised) edition 2008 \nReprinted 2008, 2009 \nISBN: 978-81-86336-14-4 \nPublisher and distributor: Yoga Publications Trust, Ganga Darshan, \nMunger, Bihar, India. \nWebsite: www .biharyoga.net \nwww . rikhiapeeth .net \nPrinted at Thomson Press (India) Limited, New Delhi, 110001 \nlV',
  'metadata': {'source_file': 'Asana.pdf',
   'doc_index': 4,
   'total_pages': 557,
   'creator': 'PFU ScanSnap Manager 4.2.14',
   'file_type': 'pdf',
   'content_length': 327,
   'creationdate': '2012-05-24T20:23:43+08:00',
   'moddate': '2021-05-07T15:35:13+02:00',
   'page_label': '5',
   'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)',
   'page': 4,
   'source': '../data/pdf/Asana.pdf'},
  'similarity_score': 0.437272310256958,
  'distance': 0.562727689743042,
  'rank': 1},
 {'id': 'doc_c6346b68_4',
  'content': 'Reprinted 2002, 2004 (twice), 2005, 2006 \nFourth (revised

### Integration Vectordb Context pipeline With LLM output


In [27]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [31]:
answer = rag_simple("what is yoga",rag_retriever,llm)
answer

Retrieving documents for query: 'what is yoga'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.92it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


"Yoga is a practice of uniting the body, thought, and breath for living in the present without suffering. It literally means 'to unite with the self' or 'become one'."